In [0]:
# Consigna 4.1: Recibir el parámetro de ADF
dbutils.widgets.text("fecha_proceso", "2024-06-25")
fecha_proceso = dbutils.widgets.get("fecha_proceso")

print(f"La fecha de proceso recibida desde Data Factory es: {fecha_proceso}")

# Variables base para las rutas
storage_account = "ccondoristad3"
landing_path = f"abfss://landing@{storage_account}.dfs.core.windows.net"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp

# Lista de los archivos que trajiste de raw a landing (Ojo: usamos incidencias_2023 por la carga inicial)
archivos = ["envios", "transportistas", "rutas", "clientes_logistica", "incidencias_2023"]

for archivo in archivos:
    print(f"Procesando {archivo} para la capa Bronze...")
    
    ruta_origen = f"{landing_path}/{archivo}/{fecha_proceso}/{archivo}.csv"
    
    # 1. Leemos solo la cabecera primero para armar el StructType dinámico (todo StringType)
    df_temp = spark.read.option("header", "true").csv(ruta_origen)
    columnas = df_temp.columns
    esquema_bronze = StructType([StructField(col, StringType(), True) for col in columnas])
    
    # 2. Leemos el archivo real aplicando el esquema estricto (Consigna 4.2)
    df_raw = (spark.read
              .option("header", "true")
              .schema(esquema_bronze)
              .csv(ruta_origen))
    
    # 3. Agregamos la columna de auditoría
    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())
    
    # 4. Guardamos en el contenedor Bronze en formato Delta
    ruta_destino = f"{bronze_path}/{archivo}"
    (df_bronze.write
     .format("delta")
     .mode("overwrite")
     .save(ruta_destino))
    
    # 5. Registramos la tabla (Como tienes configuración manual, la registramos directo en el default)
    # Reemplazamos incidencias_2023 por solo 'incidencias' para el nombre de la tabla
    nombre_tabla = archivo if archivo != "incidencias_2023" else "incidencias"
    spark.sql(f"CREATE TABLE IF NOT EXISTS bronze_{nombre_tabla} USING DELTA LOCATION '{ruta_destino}'")

print("¡Todas las tablas Bronze creadas con éxito!")

In [0]:
from pyspark.sql.functions import col, coalesce, when, lower, trim, expr

# Definimos la ruta Silver
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net"

print("Transformando la tabla envios a Silver...")

# 1. Leemos la tabla de envíos desde Bronze
df_bronze_envios = spark.read.format("delta").load(f"{bronze_path}/envios")

# 2. Aplicamos las transformaciones (Consigna 4.3)
df_silver_envios = (df_bronze_envios
    # Limpieza de fechas usando try_to_date para devolver null en caso de error
    .withColumn("fecha_envio", coalesce(
        expr("try_to_date(fecha_envio, 'yyyy-MM-dd')"),
        expr("try_to_date(fecha_envio, 'MM-dd-yyyy')"),
        expr("try_to_date(fecha_envio, 'dd/MM/yyyy')"),
        expr("try_to_date(fecha_envio, 'yyyy/MM/dd')"),
        expr("try_to_date(fecha_envio, 'yyyy/MM/dd HH:mm:ss')")
    ))
    # Limpieza de numéricos: N/A a null y cast a double
    .withColumn("peso_kg", when(col("peso_kg") == "N/A", None).otherwise(col("peso_kg")).cast("double"))
    .withColumn("monto_flete", when(col("monto_flete") == "N/A", None).otherwise(col("monto_flete")).cast("double"))
    
    # Limpieza de N/A en id_ruta
    .withColumn("id_ruta", when(col("id_ruta") == "N/A", None).otherwise(col("id_ruta")))
    
    # Limpieza de texto: minúsculas y sin espacios extra
    .withColumn("estado", lower(trim(col("estado"))))
    
    # Eliminamos duplicados por clave primaria
    .dropDuplicates(["id_envio"])
)

# 3. Guardamos en Silver
ruta_silver_envios = f"{silver_path}/envios"
(df_silver_envios.write
    .format("delta")
    .mode("overwrite")
    .save(ruta_silver_envios))

# 4. Registramos la tabla
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_envios USING DELTA LOCATION '{ruta_silver_envios}'")

print("¡Tabla silver_envios limpia y guardada!")
df_silver_envios.show(5)

In [0]:
print("Limpiando el resto de las tablas para Silver...")

# 1. Transportistas (Solo quitar duplicados)
(spark.read.format("delta").load(f"{bronze_path}/transportistas")
    .dropDuplicates(["id_transportista"])
    .write.format("delta").mode("overwrite").save(f"{silver_path}/transportistas"))
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_transportistas USING DELTA LOCATION '{silver_path}/transportistas'")

# 2. Rutas (distancia_km a número)
(spark.read.format("delta").load(f"{bronze_path}/rutas")
    .withColumn("distancia_km", col("distancia_km").cast("double"))
    .dropDuplicates(["id_ruta"])
    .write.format("delta").mode("overwrite").save(f"{silver_path}/rutas"))
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_rutas USING DELTA LOCATION '{silver_path}/rutas'")

# 3. Clientes (fecha_alta a formato fecha)
(spark.read.format("delta").load(f"{bronze_path}/clientes_logistica")
    .withColumn("fecha_alta", expr("try_cast(fecha_alta as date)"))
    .dropDuplicates(["id_cliente"])
    .write.format("delta").mode("overwrite").save(f"{silver_path}/clientes_logistica"))
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_clientes_logistica USING DELTA LOCATION '{silver_path}/clientes_logistica'")

# 4. Incidencias (fecha_incidencia a formato fecha - Carga inicial 2023)
# OJO AQUI: Leemos desde 'incidencias_2023' pero guardamos en 'incidencias'
(spark.read.format("delta").load(f"{bronze_path}/incidencias_2023")
    .withColumn("fecha_incidencia", expr("try_cast(fecha_incidencia as date)"))
    .dropDuplicates(["id_incidencia"])
    .write.format("delta").mode("overwrite").save(f"{silver_path}/incidencias"))
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_incidencias USING DELTA LOCATION '{silver_path}/incidencias'")

print("¡Las 4 tablas restantes están listas en Silver!")

In [0]:
# 1. Definir la ruta del archivo de 2024 en la capa raw
storage_account = "ccondoristad3"
raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net"
ruta_2024 = f"{raw_path}/incidencias_2024.csv"

print("Leyendo incidencias 2024 a través de Unity Catalog...")

# 2. Leer el archivo, castear fecha y AGREGAR la columna faltante
from pyspark.sql.functions import expr, current_timestamp

df_incidencias_2024 = (spark.read
    .option("header", "true")
    .csv(ruta_2024)
    .withColumn("fecha_incidencia", expr("try_cast(fecha_incidencia as date)"))
    .withColumn("ingestion_timestamp", current_timestamp()) # <- ¡Aquí está la magia!
)

# 3. Crear una vista temporal para usarla en nuestro MERGE con SQL
df_incidencias_2024.createOrReplaceTempView("vw_incidencias_2024")

print("Ejecutando MERGE INTO...")

# 4. Ejecutar el MERGE INTO
spark.sql("""
    MERGE INTO silver_incidencias AS destino
    USING vw_incidencias_2024 AS origen
    ON destino.id_incidencia = origen.id_incidencia
    WHEN MATCHED THEN
        UPDATE SET 
            destino.estado_resolucion = origen.estado_resolucion,
            destino.resolucion = origen.resolucion
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("¡MERGE INTO ejecutado con éxito!")

# 5. Verificación de Delta Time Travel
print("\n--- Historial de versiones de la tabla (Time Travel) ---")
spark.sql("DESCRIBE HISTORY silver_incidencias").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

In [0]:
# Eliminamos "abs" de este import para que no choque con el de Python
from pyspark.sql.functions import col, sum, avg, count, year, month

# Definimos la ruta de la capa Gold según los requerimientos de Fabric
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net"

print("--- INICIANDO PROCESAMIENTO DE LA CAPA GOLD (Consigna 4.5) ---")

# 1. Cargar las tablas necesarias desde la capa Silver
df_envios = spark.read.format("delta").load(f"{silver_path}/envios")
df_transportistas = spark.read.format("delta").load(f"{silver_path}/transportistas")
df_rutas = spark.read.format("delta").load(f"{silver_path}/rutas")

# 2. Extraer Año y Mes de la columna fecha_envio para las agrupaciones
df_envios_fechas = (df_envios
    .withColumn("anio", year(col("fecha_envio")))
    .withColumn("mes", month(col("fecha_envio")))
)


# === TABLA GOLD 1: gold_envios_zona ===
print("Procesando: gold_envios_zona...")
df_gold_zona = (df_envios_fechas
    .join(df_transportistas, on="id_transportista", how="inner")
    .groupBy("zona", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        sum("monto_flete").alias("monto_total"),
        sum("peso_kg").alias("peso_total"),
        (sum("monto_flete") / count("id_envio")).alias("ticket_promedio")
    )
)

# Guardar en ADLS Gen2 y registrar en Unity Catalog
ruta_gold_zona = f"{gold_path}/envios_zona"
df_gold_zona.write.format("delta").mode("overwrite").save(ruta_gold_zona)
spark.sql(f"CREATE TABLE IF NOT EXISTS gold_envios_zona USING DELTA LOCATION '{ruta_gold_zona}'")


# === TABLA GOLD 2: gold_envios_tipo_ruta ===
print("Procesando: gold_envios_tipo_ruta...")
df_gold_tipo_ruta = (df_envios_fechas
    .join(df_rutas, on="id_ruta", how="inner")
    .groupBy("tipo_ruta", "anio", "mes")
    .agg(
        count("id_envio").alias("total_envios"),
        sum("monto_flete").alias("monto_total"),
        (sum("monto_flete") / count("id_envio")).alias("ticket_promedio"),
        avg("distancia_km").alias("distancia_promedio")
    )
)

# Guardar en ADLS Gen2 y registrar en Unity Catalog
ruta_gold_tipo_ruta = f"{gold_path}/envios_tipo_ruta"
df_gold_tipo_ruta.write.format("delta").mode("overwrite").save(ruta_gold_tipo_ruta)
spark.sql(f"CREATE TABLE IF NOT EXISTS gold_envios_tipo_ruta USING DELTA LOCATION '{ruta_gold_tipo_ruta}'")


# === SECCIÓN DE VERIFICACIÓN (Obligatoria según la rúbrica) ===
print("\n--- Ejecutando Validación de Consistencia ---")
total_filas_silver = df_envios.count()
suma_total_envios_gold_zona = df_gold_zona.agg(sum("total_envios")).collect()[0][0]

# Aquí ya usa el abs nativo de Python perfectamente
import builtins
diferencia = builtins.abs(total_filas_silver - suma_total_envios_gold_zona)


print(f"Total registros en 'silver_envios': {total_filas_silver}")
print(f"Suma de 'total_envios' en 'gold_envios_zona': {suma_total_envios_gold_zona}")
print(f"Diferencia calculada: {diferencia} registros.")

if diferencia <= 10:
    print("¡VALIDACIÓN EXITOSA! La diferencia está dentro de la tolerancia de 10 filas por envíos sin transportista.")
else:
    print("ALERTA: La diferencia supera la tolerancia permitida de 10 filas. Revisa los joins.")

print("\n--- Vista previa de gold_envios_zona ---")
df_gold_zona.show(5)

In [0]:
# === Consigna 4.6: Celda de limpieza (Comentada por defecto) ===
# print("Eliminando tablas de Unity Catalog...")

# # Tablas Capa Bronze
# spark.sql("DROP TABLE IF EXISTS default.bronze_envios")
# spark.sql("DROP TABLE IF EXISTS default.bronze_transportistas")
# spark.sql("DROP TABLE IF EXISTS default.bronze_rutas")
# spark.sql("DROP TABLE IF EXISTS default.bronze_clientes_logistica")
# spark.sql("DROP TABLE IF EXISTS default.bronze_incidencias")

# # Tablas Capa Silver
# spark.sql("DROP TABLE IF EXISTS default.silver_envios")
# spark.sql("DROP TABLE IF EXISTS default.silver_transportistas")
# spark.sql("DROP TABLE IF EXISTS default.silver_rutas")
# spark.sql("DROP TABLE IF EXISTS default.silver_clientes_logistica")
# spark.sql("DROP TABLE IF EXISTS default.silver_incidencias")

# # Tablas Capa Gold
# spark.sql("DROP TABLE IF EXISTS default.gold_envios_zona")
# spark.sql("DROP TABLE IF EXISTS default.gold_envios_tipo_ruta")

# print("¡Limpieza completada!")